In [ ]:
import sys
import torch
import time
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

def quantize_per_vector(tensor, mantissa_bits, dim):
    """Quantizes a tensor per-vector along a specified dimension."""
    max_quant_val = (2**mantissa_bits) - 1
    # Calculate scaling factors (sf) per vector along the specified dimension
    max_abs_vals = torch.max(torch.abs(tensor), dim=dim, keepdim=True).values
    sf = torch.where(max_abs_vals > 0, max_abs_vals / max_quant_val, 1.0)
    sf = sf.to(torch.float8_e4m3fn).to(tensor.dtype)
    # In-place clamp to avoid division by zero or very small numbers
    sf.clamp_(min=1e-12)
    # Quantize by dividing by the scaling factor
    quantized_tensor = torch.round(tensor / sf)
    return quantized_tensor, sf

def add_adc_noise(tensor, mean, sigma, lsb):
    """Adds simulated Analog-to-Digital Converter (ADC) noise to the tensor using efficient in-place operations."""
    if sigma <= 0:
        return tensor
    # Create noise tensor and perform calculations in-place to save memory
    noise_in_volts = torch.randn(tensor.size(), dtype=tensor.dtype, device=tensor.device)
    noise_in_volts.mul_(sigma).add_(mean) # In-place multiplication and addition
    integer_noise = noise_in_volts.div_(lsb).round_() # In-place division and rounding

    # Add noise to the original tensor in-place
    return tensor.add_(integer_noise)

class SimulatedFFN(nn.Module):
    """A simulated Feed-Forward Network that models tiled hardware with quantization and noise."""
    def __init__(self, original_mlp, tile_shape=(32, 40), noise_mean=0.0, noise_sigma=0.001):
        super().__init__()
        # Extract weights from the original FFN layer
        self.gate_proj = original_mlp.gate_proj
        self.up_proj = original_mlp.up_proj
        self.down_proj = original_mlp.down_proj

        # Simulation parameters
        v_min, v_max, levels = -0.5, 0.5, 256
        self.lsb = (v_max - v_min) / levels
        self.noise_mean = noise_mean
        self.noise_sigma = noise_sigma
        self.tile_shape = tile_shape

    @torch.compile(fullgraph=True)
    def tiled_mat_mul_vec(self, input_tensor, weight_tensor):
        weight_tensor = weight_tensor.T
        m, k = input_tensor.shape
        _, n = weight_tensor.shape
        tile_m, block_size = self.tile_shape
        tile_n = tile_m

        # Pad tensors to be divisible by tile dimensions
        pad_m = (tile_m - m % tile_m) % tile_m
        pad_k = (block_size - k % block_size) % block_size
        pad_n = (tile_n - n % tile_n) % tile_n

        inp_pad = nn.functional.pad(input_tensor, (0, pad_k, 0, pad_m))
        w_pad = nn.functional.pad(weight_tensor, (0, pad_n, 0, pad_k))
        m_pad, k_pad = inp_pad.shape
        _, n_pad = w_pad.shape

        # Create unfolded views (tiles) of the input and weight tensors
        inp_tiles = inp_pad.unfold(0, tile_m, tile_m).unfold(1, block_size, block_size)
        w_tiles = w_pad.unfold(0, block_size, block_size).unfold(1, tile_n, tile_n)

        # Quantize each tile
        inp_quant_tiles, inp_sf_tiles = quantize_per_vector(inp_tiles, mantissa_bits=7, dim=3)
        w_quant_tiles, w_sf_tiles = quantize_per_vector(w_tiles, mantissa_bits=2, dim=2)

        # Perform matmul on integer mantissas
        accum_mantissas = torch.matmul(
            inp_quant_tiles.unsqueeze(2).to(torch.bfloat16),
            w_quant_tiles.unsqueeze(0).to(torch.bfloat16)
        )

        # Apply noise to the accumulated mantissa products
        noisy_accum_mantissas = add_adc_noise(accum_mantissas, self.noise_mean, self.noise_sigma, self.lsb)

        # Dequantize the noisy results
        combined_sf = inp_sf_tiles.unsqueeze(2) * w_sf_tiles.unsqueeze(0)

        # In-place multiplication to save memory on the large dequantized_tiles tensor
        dequantized_tiles = noisy_accum_mantissas.mul_(combined_sf)

        # Accumulate the dequantized tile outputs
        output_tiles = torch.sum(dequantized_tiles, dim=1).to(input_tensor.dtype)

        # Reshape output tiles back to a 2D matrix
        out_pad = output_tiles.permute(0, 2, 1, 3).contiguous().view(m_pad, n_pad)

        return out_pad[:m, :n]

    def forward(self, x):
        x_2d = x.view(-1, x.shape[-1])

        gated_states = self.gate_proj(x_2d)
        nn.functional.silu(gated_states, inplace=True)

        up_states = self.tiled_mat_mul_vec(x_2d, self.up_proj.weight)

        # In-place element-wise multiplication
        activated_states = gated_states.mul_(up_states)

        output_states_2d = self.tiled_mat_mul_vec(activated_states, self.down_proj.weight)
        return output_states_2d.view(x.shape)

def replace_ffn_with_simulation(model, noise_mean, noise_sigma):
    """Replaces all FFN layers in the model with the SimulatedFFN."""
    for layer in model.model.layers:
        original_mlp = layer.mlp
        layer.mlp = SimulatedFFN(original_mlp, noise_mean=noise_mean, noise_sigma=noise_sigma)
    print("Replacement complete.")
    return model

@torch.no_grad()
def evaluate_hellaswag(model, tokenizer, device):
    """Evaluates the model on the Hellaswag dataset by calculating completion likelihood."""
    print("Loading Hellaswag dataset for evaluation...")
    hellaswag_dataset = load_dataset("hellaswag", split="validation")

    total_correct = 0
    total_samples = 0
    tokenizer.add_bos_token = False

    for sample in tqdm(hellaswag_dataset, desc="Evaluating Hellaswag"):
        context = sample['ctx']
        endings = sample['endings']
        correct_label = int(sample['label'])

        completion_log_likelihoods = []
        context_tokens = tokenizer(context, return_tensors='pt', add_special_tokens=False).to(device)
        context_len = context_tokens.input_ids.shape[1]

        for ending in endings:
            full_text = context + " " + ending
            full_tokens = tokenizer(full_text, return_tensors='pt', add_special_tokens=False).to(device)
            full_ids = full_tokens.input_ids

            outputs = model(full_ids)
            logits = outputs.logits

            ending_logits = logits[:, context_len - 1:-1, :]
            ending_ids = full_ids[:, context_len:]

            log_probs = torch.nn.functional.log_softmax(ending_logits, dim=-1)
            token_log_probs = log_probs.gather(dim=-1, index=ending_ids.unsqueeze(-1)).squeeze(-1)
            score = torch.sum(token_log_probs)
            completion_log_likelihoods.append(score.item())

        if np.argmax(completion_log_likelihoods) == correct_label:
            total_correct += 1
        total_samples += 1

    accuracy = total_correct / total_samples
    print(f"\nHellaswag Overall Accuracy: {accuracy:.4f}")
    return {"hellaswag_accuracy": accuracy}

In [ ]:
class Logger:
    """A simple logger to write output to both console and a file."""
    def __init__(self, filepath, stream):
        self.stream = stream
        self.log = open(filepath, 'a', buffering=1)

    def write(self, message):
        self.stream.write(message)
        self.log.write(message)

    def flush(self):
        self.stream.flush()
        self.log.flush()

    def __del__(self):
        self.log.close()

In [ ]:
# --- Main Execution ---
if __name__ == "__main__":
    MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
    out_path = "./drive/Shareddrives/MXPTQ"

    NOISE_MEAN = 0.0
    NOISE_SIGMA = 0.001

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log_file_path = f"{out_path}/llama_quant_hw_hellaswag_output_efficient.log"

    original_stdout = sys.stdout
    original_stderr = sys.stderr
    # Uncomment the following lines to enable file logging in a non-interactive environment
    # sys.stdout = Logger(log_file_path, original_stdout)
    # sys.stderr = Logger(log_file_path, original_stderr)

    try:
        print(f"Using device: {device}")
        print("Loading base model and tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            dtype=torch.bfloat16,
            device_map=device,
        )
        model.eval()

        print("Running Quantization with HW Noise Run")
        quant_hw_model = replace_ffn_with_simulation(model, noise_mean=NOISE_MEAN, noise_sigma=NOISE_SIGMA)

        hellaswag_results = evaluate_hellaswag(quant_hw_model, tokenizer, device)

        torch.save(hellaswag_results, f"{out_path}/llama_quant_hw_hellaswag_results_efficient.pt")
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        print("--- Logging restored ---")
        print(f"Hellaswag results saved to {out_path}/llama_quant_hw_hellaswag_results_efficient.pt")